In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import NesterovMomentumOptimizer
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from pennylane.optimize import AdamOptimizer
from pennylane import numpy as pnp
import os

data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target
print(df.shape)
df.head()

In [ ]:
'''print(data.target_names)
print(df['target'].value_counts())'''


In [ ]:
'''print(df.isnull().sum().sum())'''

In [ ]:
'''print(data.feature_names)
df.describe()'''

In [ ]:
'''malignant = df[df['target'] == 0]['mean radius']
benign = df[df['target'] == 1]['mean radius']

plt.hist(malignant, bins=30, alpha=0.5, label='malignant', color='red')
plt.hist(benign, bins=30, alpha=0.5, label='benign', color='green')
plt.legend()
plt.xlabel('mean radius')
plt.show()'''
                        

In [ ]:
'''corr = df.drop('target', axis=1).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='RdYlGn', center=0)
plt.show()'''

In [ ]:
scaler = StandardScaler()
X = df.drop('target', axis=1).values
y = df['target'].values

X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

print(pca.explained_variance_ratio_)
print(f"Total variance kept: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(
    X_pca, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape)
print(X_test.shape)

#INITIAL PARAMETERS
n_qubits = 4
n_layers = 25
weights = pnp.random.uniform(0, np.pi, (n_layers, n_qubits, 3), requires_grad=True)
weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
x = np.random.uniform(0, np.pi, n_qubits)

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def full_pipeline(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))
print(full_pipeline(x, weights))

def classifier(weights, bias, x):
    return full_pipeline(x, weights) + bias

def square_loss(labels, predictions):
    preds = qml.math.stack(predictions)
    return qml.math.sum((labels - preds) ** 2) / len(labels)

def cost(weights, bias, X, y):
    preds = [classifier(weights, bias, x) for x in X]
    return square_loss(y, preds)

def accuracy(labels, weights, bias, X):
    preds = np.sign(np.array([float(classifier(weights, bias, x)) for x in X]))
    return np.mean(np.array(labels) == preds)

label = 1.0
prediction = 0.40

opt = AdamOptimizer(stepsize=0.1)
weights = pnp.random.uniform(0, pnp.pi, weights_shape, requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)

print(weights.shape)
print(bias)

X_sample = pnp.array(X_train[:5], requires_grad=False)
y_sample = pnp.array(y_train[:5], requires_grad=False)
print(cost(weights, bias, X_sample, y_sample))

In [ ]:
#STRONGLY ENTANGLING LAYERS
import numpy as np
X_tr = pnp.array(X_train, requires_grad=False)
Y_tr = pnp.array(2 * y_train - 1, requires_grad=False)
X_te = pnp.array(X_test, requires_grad=False)
Y_te = pnp.array(2 * y_test - 1, requires_grad=False)

strong_accs = []
strong_best_acc = []
strong_costs = []
strong_best_cost = []
strong_history = []

pnp.random.seed(42)
weights = pnp.array(np.random.uniform(0, np.pi, weights_shape), requires_grad=True)
weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
bias = pnp.array(0.0, requires_grad=True)
opt = AdamOptimizer(0.1)

n_qubits = 4
n_layers = 25
n_pca = 4
batch_size = 20
n_epochs = 100
best_acc = 0.0
best_cost = 0.0

for epoch in range(n_epochs):
    idx = pnp.random.randint(0, len(X_tr), (batch_size,))
    result = opt.step(cost, weights, bias, X_tr[idx], Y_tr[idx])
    weights, bias = result[0], result[1]
    c = cost(weights, bias, X_tr, Y_tr)
    strong_costs.append(c)
    acc = accuracy(Y_te, weights, bias, X_te)
    strong_accs.append(acc)
    print(f"Epoch {epoch+1:3d} | Cost: {c:.4f} | Test Acc: {acc*100:.1f}% | Bias: {bias:.4f}")
    #print(f"{weights.reshape(24, 5)}")
    if acc > best_acc:
        best_acc = acc
    if c < best_cost or best_cost == 0.0:
        best_cost = c
    strong_history.append({
        'Model_type': 'StronglyEntanglingLayers',
        'Epoch': epoch + 1,
        'Qubits': n_qubits,
        'Layers': n_layers,
        'PCA Layers': n_pca,
        'Total Epochs': n_epochs,
        'Cost': float(c),
        'Best Cost': float(best_cost),
        'Accuracy': float(acc),
        'Best Accuracy': float(best_acc)
    })
print(f"Best Test Accuracy: {best_acc*100:.1f}%")
print(f"Best Test Cost: {best_cost:.4f}")
a_best_acc = best_acc
a_best_cost = best_cost
strong_best_acc.append(best_acc)
strong_best_cost.append(best_cost)

os.makedirs("Outputs", exist_ok=True)
df_strong_history = pd.DataFrame(strong_history)
df_strong_history.to_csv('Outputs/02_Output_StrongHistory.csv', index=False)

In [ ]:
#BASIC ENTANGLEMENT
@qml.qnode(dev)
def full_pipeline(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return qml.expval(qml.PauliZ(0))

X_tr = pnp.array(X_train, requires_grad=False)
Y_tr = pnp.array(2 * y_train - 1, requires_grad=False)
X_te = pnp.array(X_test, requires_grad=False)
Y_te = pnp.array(2 * y_test - 1, requires_grad=False)

pnp.random.seed(42)
weights = pnp.array(np.random.uniform(0, np.pi, weights_shape), requires_grad=True)
weights_shape = qml.BasicEntanglerLayers.shape(n_layers=n_layers, n_wires=n_qubits)
bias = pnp.array(0.0, requires_grad=True)
opt = AdamOptimizer(0.1)


n_qubits = 4
n_layers = 25
n_pca = 4
batch_size = 20
n_epochs = 100
best_acc = 0.0
best_cost = 0.0

basic_accs = []
basic_best_acc = []
basic_costs = []
basic_best_cost = []
basic_history = []


for epoch in range(n_epochs):
    idx = pnp.random.randint(0, len(X_tr), (batch_size,))
    result = opt.step(cost, weights, bias, X_tr[idx], Y_tr[idx])
    weights, bias = result[0], result[1]
    c = cost(weights, bias, X_tr, Y_tr)
    basic_costs.append(c)
    acc = accuracy(Y_te, weights, bias, X_te)
    basic_accs.append(acc)
    print(f"Epoch {epoch+1:3d} | Cost: {c:.4f} | Test Acc: {acc*100:.1f}% | Bias: {bias:.4f}")
    #print(f"{weights.reshape(8, 5)}")
    if acc > best_acc:
        best_acc = acc
    if c < best_cost or best_cost == 0.0:
        best_cost = c
    basic_history.append({
        'Model_type': 'BasicEntanglerLayers',
        'Epoch': epoch + 1,
        'Qubits': n_qubits,
        'Layers': n_layers,
        'PCA Layers': n_pca,
        'Total Epochs': n_epochs,
        'Cost': float(c),
        'Best Cost': float(best_cost),
        'Accuracy': float(acc),
        'Best Accuracy': float(best_acc)
    })
os.makedirs("Outputs", exist_ok=True)
print(f"Best Test Accuracy: {best_acc*100:.1f}%")
print(f"Best Test Cost: {best_cost:.4f}")
b_best_acc = best_acc
b_best_cost = best_cost
basic_best_acc.append(best_acc)
basic_best_cost.append(best_cost)    

os.makedirs("Outputs", exist_ok=True)
df_basic_history = pd.DataFrame(basic_history)
df_basic_history.to_csv('Outputs/02_Output_BasicHistory.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Use the pre-existing lists tracking accuracy history
epochs = range(1, len(strong_accs) + 1)
ax.plot(epochs, strong_costs, marker='o', linewidth=2, label='Strongly Entangling Layers', color='royalblue')
ax.plot(epochs, basic_costs, marker='s', linewidth=2, label='Basic Entangling Layers', color='darkorange')

# Formatting the plot
ax.set_title('Test Cost Over Epochs', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Cost', fontsize=12)

# Ensure the x-axis displays integer ticks for each epoch (e.g., 1, 2, 3)
ax.set_xticks(epochs)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

# Use the pre-existing lists tracking accuracy history
epochs = range(1, len(strong_accs) + 1)
ax.plot(epochs, strong_accs, marker='o', linewidth=2, label='Strongly Entangling Layers', color='royalblue')
ax.plot(epochs, basic_accs, marker='s', linewidth=2, label='Basic Entangling Layers', color='darkorange')

# Formatting the plot
ax.set_title('Test Accuracy Over Epochs', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=10)

# Ensure the x-axis displays integer ticks for each epoch (e.g., 1, 2, 3)
ax.set_xticks(epochs)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(fontsize=11)

plt.show()

In [ ]:
acc_diff = pd.Series(np.array(strong_accs) - np.array(basic_accs))

fig, ax = plt.subplots(figsize=(8, 5))
acc_diff.plot(kind='bar', ax=ax, color='royalblue', edgecolor='black')

ax.set_title('Accuracy Difference Between Strong Entangling and Basic Entangling Models per Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy Difference')

# Change x-axis labels from 0,1,2,3,4 to 1,2,3,4,5
ax.set_xticklabels([f"{i+1}" for i in range(len(acc_diff))], rotation=0)

plt.tight_layout()
plt.show()